# 01 — Segmentation smoke test

Goal: confirm that CellPose installs correctly on Apple Silicon and can segment a simple image end-to-end on this machine.

This notebook doesn't use any project data. It runs CellPose's built-in `cyto2` model on a small synthetic test image and renders the segmentation overlay. If this notebook runs cleanly, the segmentation step of the Maillat 2026 pipeline (section 6.1) is technically viable on my hardware.

In [ ]:
# Imports for segmentation smoke test
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models
from skimage import data

# Make sure inline plotting works in this notebook
%matplotlib inline

print("Imports successful")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Install the missing scikit-image data downloader
!pip install pooch

In [ ]:
# Use a built-in scikit-image test image that doesn't need pooch
# data.coins() is small, single-channel, and ships with the package itself
from skimage import data

img_2d = data.coins()
print(f"Image shape: {img_2d.shape}")
print(f"Pixel value range: {img_2d.min()} to {img_2d.max()}")
print(f"Data type: {img_2d.dtype}")

# Show what we're about to segment
plt.figure(figsize=(6, 6))
plt.imshow(img_2d, cmap='gray')
plt.title('Test image: data.coins() — ships with scikit-image')
plt.axis('off')
plt.show()

In [ ]:
# Run CellPose 4's default model (Cellpose-SAM / cpsam)
# Note: Maillat 2026 uses cyto2 from CellPose 3.x. We have CellPose 4.x
# which renamed the class and changed the default model. We'll log this
# deviation in docs/methods_notes.md.

# gpu=False keeps things simple on Apple Silicon for the smoke test;
# we'll enable MPS GPU later when working with real data
model = models.CellposeModel(gpu=False)

# In CellPose 4, channels param is gone (model is channel-invariant)
# diameter=None lets the model estimate cell size on its own
results = model.eval(img_2d, diameter=None)
masks = results[0]  # first element is always the mask array

n_cells = len(np.unique(masks)) - 1  # subtract 1 for the background
print(f"Number of objects segmented: {n_cells}")
print(f"Mask shape: {masks.shape}")
print(f"Mask dtype: {masks.dtype}")

In [ ]:
# Visualize the segmentation result
from skimage.color import label2rgb

# Side-by-side: original on the left, segmentation overlay on the right
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(img_2d, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

# label2rgb colors each segmented object differently and overlays on the image
overlay = label2rgb(masks, image=img_2d, bg_label=0, alpha=0.4)
axes[1].imshow(overlay)
axes[1].set_title(f'CellPose-SAM segmentation ({n_cells} objects found)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/01_smoke_test_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved to results/figures/01_smoke_test_segmentation.png")

## Conclusion

The segmentation pipeline runs end-to-end on this machine:

- **Environment**: Apple Silicon (arm64) macOS, Python 3.10, CellPose 4.1.1, PyTorch 2.12 (CPU)
- **Model**: Cellpose-SAM (`cpsam`, the default in CellPose 4.x)
- **Test input**: `skimage.data.coins()` — 24 ancient Greek coins
- **Result**: 24/24 objects segmented, mask shape (303, 384), saved overlay to `results/figures/01_smoke_test_segmentation.png`

### Deviations from Maillat 2026

The paper uses **CellPose 3.x with the `cyto2` model**. We're on **CellPose 4.x with `cpsam`** because that's the current default install. Two paths forward when we move to real data:

1. **Pin CellPose to 3.x** in `requirements.txt` for maximum paper-fidelity
2. **Stay on 4.x** and document this as a methods deviation

Decision deferred until we have real cell images to compare on.